In [5]:
from pyspark.sql import SparkSession
from dotenv import load_dotenv
import os
load_dotenv()
print(os.getenv("SPARK_LOCAL_IP"))

127.0.0.1


In [6]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-6")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


**Task 1**

From orders.csv, use withColumn() to add three new columns: revenue (unit_price * quantity),

 discounted_price (unit_price * (1 - discount_pct / 100)), and is_delivered (a boolean — true when status equals "Delivered"). 

Show all three new columns alongside order_id.

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
orders = spark.read.csv(
    "s3a://pyspark-30-days-rahul-2026/data/orders.csv",
    header=True,
    inferSchema=True
)

orders.withColumn(
    "revenue",
    col("unit_price") * col("quantity")
).withColumn(
    "discounted_price",
    col("unit_price") * (1 - col("discount_pct") / 100)
).withColumn(
    "is_delivered",
    col("status") == "delivered"
).select(col('order_id'),
col('revenue'),
col('discounted_price'),
col('is_delivered')).show(4)

+--------+-------+----------------+------------+
|order_id|revenue|discounted_price|is_delivered|
+--------+-------+----------------+------------+
|   O0001|2599.98|        1169.991|       false|
|   O0002| 449.99|          449.99|       false|
|   O0003|1399.96|        297.4915|       false|
|   O0004| 179.98|         85.4905|       false|
+--------+-------+----------------+------------+
only showing top 4 rows


**Task 2**

Use withColumnRenamed() to rename unit_price to price and customer_id to cust_id.

 Print the column list to verify.

In [8]:
renamed_orders = orders.withColumnRenamed("unit_price", "price") \
    .withColumnRenamed("customer_id", "cust_id")
renamed_orders.columns

['order_id',
 'cust_id',
 'product_id',
 'order_date',
 'quantity',
 'price',
 'discount_pct',
 'status',
 'payment_method',
 'region']

**Task 3**

From customers.csv, drop the columns email, country, and signup_date.

 How many columns remain? Print them.

In [9]:
customers=spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/customers.csv",header=True,inferSchema=True)
dropped_customers=customers.drop("email","country","signup_date")
dropped_customers.columns

['customer_id', 'first_name', 'last_name', 'city', 'state', 'segment']

**Task 4**

Using orders.csv, add a revenue column, then try dropping a column that does not exist — like nonexistent_col.

What happens? Does it throw an error?

In [10]:
orders.withColumn("revenue",F.col("unit_price") * F.col("quantity")).show(4)
orders.drop('nonexistent_column').show(4) 

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|revenue|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|2599.98|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West| 449.99|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|          15|Delivered|   Credit Card|Midwest|1399.96|
|   O0004|       C004|      P006|2023-01-12|       2|     89.99|           5|Delivered|    Debit Card|  South| 179.98|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-------+
only showing top 4 rows


+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|          15|Delivered|   Credit Card|Midwest|
|   O0004|       C004|      P006|2023-01-12|       2|     89.99|           5|Delivered|    Debit Card|  South|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
only showing top 4 rows


drop() is designed to be idempotent. If the specified column exists, Spark removes it. If it doesn't exist, Spark simply returns the original DataFrame without throwing an exception. This makes schema modification operations safe and avoids unnecessary failures.

**Rule to remember**

Transformations that only modify the schema (like drop() and withColumnRenamed()) are generally idempotent—if the target column doesn't exist, Spark leaves the DataFrame unchanged.


Transformations that need to read column data (like select(), filter(), withColumn()) must resolve the column, so they throw an AnalysisException if it's missing.